# 🛒 Semana 09: Segmentación de Clientes - Consumo Masivo de Alimentos
## Dataset: Wholesale Customer (Clientes de distribución de alimentos)

**Objetivo:** Segmentar clientes de una empresa de distribución de alimentos basándose en sus patrones de compra.

**Algoritmos a competir:**
- **K-Means**: Clustering por centroides
- **DBSCAN**: Clustering por densidad (detecta outliers)
- **Clustering Jerárquico**: Dendrograma y clusters anidados
- **PCA**: Reducción a 2D para visualizar clusters
- **t-SNE**: Visualización avanzada para alta dimensionalidad

**Métricas de evaluación:**
- Silhouette Score (qué tan bien separados están los clusters)
- Inercia (distancia intra-cluster - solo K-Means)
- Dendrograma (para clustering jerárquico)
- Análisis de perfiles de cliente por cluster

**Contexto de negocio:** Una empresa de distribución de alimentos quiere segmentar sus clientes para:
- Ofrecer promociones personalizadas
- Identificar clientes de alto valor
- Optimizar estrategias de marketing

---

### ¿Qué es?
Dataset de **440 clientes** de una empresa de distribución mayorista de alimentos. Contiene información de gasto anual (en miles de unidades monetarias) en diferentes categorías:

**Características:**
- **Channel**: Canal de venta (1=Hoteles/Restaurantes/Cafés, 2=Minorista)
- **Region**: Región geográfica (1=Lisboa, 2=Oporto, 3=Otras)
- **Fresh**: Gasto anual en productos frescos
- **Milk**: Gasto anual en productos lácteos
- **Grocery**: Gasto anual en abarrotes
- **Frozen**: Gasto anual en productos congelados
- **Detergents_Paper**: Gasto anual en detergentes y papel
- **Delicassen**: Gasto anual en delicatessen

### ¿Qué problema resuelve?
Segmentar clientes para entender sus patrones de compra y diseñar estrategias comerciales personalizadas.

### El Reto
**Segmentación no supervisada** - El algoritmo debe descubrir por sí mismo los grupos de clientes basándose en sus patrones de gasto.

### Contexto de negocio
Una empresa de distribución de alimentos quiere segmentar sus clientes para ofrecer promociones personalizadas y optimizar su estrategia de marketing.

## 1. Configuración Inicial

Importamos las librerías necesarias y configuramos la semilla para reproducibilidad.

In [ ]:
# ======================================================
# SEMANA 09: WHOLESALE CUSTOMER SEGMENTATION COMPLETO
# ======================================================

# Instalar librerías necesarias
!pip install scikit-learn pandas numpy matplotlib seaborn -q

# Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Configuración
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("✅ Librerías importadas correctamente")

## 2. Carga y Exploración de Datos

Cargamos el dataset Wholesale Customer desde UCI (online).

In [ ]:
print("="*60)
print("📊 CARGANDO DATASET WHOLESALE CUSTOMER")
print("="*60)

# URL del dataset (UCI Machine Learning Repository)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00292/Wholesale%20customers%20data.csv"
df = pd.read_csv(url)

# Verificar nombres de columnas
print("\n📋 Columnas del dataset:")
print(df.columns.tolist())

# Renombrar columna 'Delicassen' si existe (error común)
if 'Delicassen' in df.columns:
    df = df.rename(columns={'Delicassen': 'Delicatessen'})

print(f"\n✅ Dataset cargado exitosamente")
print(f"📊 SHAPE: {df.shape[0]:,} clientes × {df.shape[1]} características")

# Mostrar primeras filas
print("\n📋 Primeras 5 filas del dataset:")
display(df.head())

# Estadísticas descriptivas
print("\n📊 Estadísticas descriptivas:")
display(df.describe())

In [ ]:
# Análisis exploratorio
print("="*60)
print("📈 ANÁLISIS EXPLORATORIO")
print("="*60)

# Distribución por canal
channel_map = {1: 'HORECA (Hoteles/Restaurantes/Cafés)', 2: 'Minorista'}
df['Channel_Name'] = df['Channel'].map(channel_map)

print("\n📊 Distribución por canal de venta:")
channel_counts = df['Channel_Name'].value_counts()
for channel, count in channel_counts.items():
    print(f"   • {channel}: {count} clientes ({count/len(df)*100:.1f}%)")

# Distribución por región
region_map = {1: 'Lisboa', 2: 'Oporto', 3: 'Otras'}
df['Region_Name'] = df['Region'].map(region_map)

print("\n📊 Distribución por región:")
region_counts = df['Region_Name'].value_counts()
for region, count in region_counts.items():
    print(f"   • {region}: {count} clientes ({count/len(df)*100:.1f}%)")

# Categorías de productos (excluyendo Channel y Region)
categories = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicatessen']

# Visualización de gasto por categoría
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for i, cat in enumerate(categories):
    axes[i].hist(df[cat], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[i].set_xlabel(f'Gasto anual en {cat}')
    axes[i].set_ylabel('Frecuencia')
    axes[i].set_title(f'Distribución - {cat}')
    axes[i].grid(axis='y')

plt.tight_layout()
plt.show()

# Matriz de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(df[categories].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Matriz de Correlación - Categorías de Productos')
plt.tight_layout()
plt.show()

print("\n💡 Observación: Grocery, Milk y Detergents_Paper están altamente correlacionados.")
print("   Esto sugiere que clientes que compran uno tienden a comprar los otros.")

## 3. Preprocesamiento

Preparamos los datos para clustering: escalado y selección de características.

In [ ]:
print("="*60)
print("🔄 PREPROCESAMIENTO")
print("="*60)

# Seleccionar características para clustering (excluyendo Channel y Region)
features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicatessen']
X = df[features].copy()

# Escalar los datos (importante para K-Means, DBSCAN y distancia euclidiana)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"📊 Características seleccionadas: {features}")
print(f"📊 Shape después de escalado: {X_scaled.shape}")

# PCA para visualización
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"\n📊 Varianza explicada por PCA:")
print(f"   • PC1: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"   • PC2: {pca.explained_variance_ratio_[1]*100:.2f}%")
print(f"   • Total: {pca.explained_variance_ratio_.sum()*100:.2f}%")

# t-SNE para visualización avanzada
print("\n⏳ Calculando t-SNE (puede tomar unos segundos)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

# Visualización PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=50, c='steelblue', edgecolors='k')
axes[0].set_xlabel('Componente Principal 1')
axes[0].set_ylabel('Componente Principal 2')
axes[0].set_title('Visualización con PCA')
axes[0].grid(True)

axes[1].scatter(X_tsne[:, 0], X_tsne[:, 1], alpha=0.6, s=50, c='steelblue', edgecolors='k')
axes[1].set_xlabel('t-SNE Componente 1')
axes[1].set_ylabel('t-SNE Componente 2')
axes[1].set_title('Visualización con t-SNE')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 4. K-Means Clustering

K-Means agrupa clientes en k segmentos basados en sus patrones de gasto.

In [ ]:
print("="*60)
print("📊 K-MEANS CLUSTERING")
print("="*60)

# Método del codo para encontrar el k óptimo
inertias = []
silhouettes = []
k_range = range(2, 11)

print("\n🔍 Calculando métricas para diferentes valores de k...")
for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)
    sil_score = silhouette_score(X_scaled, kmeans.labels_)
    silhouettes.append(sil_score)
    print(f"   k={k}: Inercia={kmeans.inertia_:.0f}, Silhouette={sil_score:.4f}")

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].set_title('Método del Codo - K-Means')
axes[0].grid(True)

axes[1].plot(k_range, silhouettes, 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score - K-Means (mayor es mejor)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Elegir k óptimo (el que maximiza silhouette)
best_k = k_range[np.argmax(silhouettes)]
print(f"\n✅ Mejor k según Silhouette Score: {best_k}")

# Entrenar K-Means con el mejor k
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
df['Segment_KMeans'] = kmeans_labels

final_silhouette = silhouette_score(X_scaled, kmeans_labels)
print(f"\n📊 Resultados de K-Means (k={best_k}):")
print(f"   • Inercia: {kmeans.inertia_:.0f}")
print(f"   • Silhouette Score: {final_silhouette:.4f}")

# Distribución de clientes por segmento
print(f"\n📊 Distribución de clientes por segmento:")
segment_counts = df['Segment_KMeans'].value_counts().sort_index()
for seg, count in segment_counts.items():
    print(f"   • Segmento {seg}: {count} clientes ({count/len(df)*100:.1f}%)")

## 5. Perfiles de los Segmentos (K-Means)

Analizamos las características de cada segmento para entender los perfiles de clientes.

In [ ]:
print("="*60)
print("📊 PERFILES DE SEGMENTOS (K-MEANS)")
print("="*60)

# Calcular promedios por segmento
segment_profiles = df.groupby('Segment_KMeans')[features].mean()

print("\n📊 Gasto promedio por categoría (en unidades monetarias):")
display(segment_profiles.round(0).astype(int))

# Visualización de perfiles
fig, ax = plt.subplots(figsize=(12, 6))

# Normalizar para comparación
segment_profiles_norm = (segment_profiles - segment_profiles.min()) / (segment_profiles.max() - segment_profiles.min())
segment_profiles_norm.T.plot(kind='bar', ax=ax, colormap='tab10')
ax.set_xlabel('Categoría de Producto')
ax.set_ylabel('Gasto normalizado')
ax.set_title('Perfil de Gasto por Segmento (K-Means)')
ax.legend(title='Segmento')
ax.grid(axis='y')
plt.tight_layout()
plt.show()

# Interpretación de segmentos
print("\n📖 INTERPRETACIÓN DE SEGMENTOS:")
for seg in range(best_k):
    if seg in segment_profiles.index:
        top_categories = segment_profiles.loc[seg].nlargest(3).index.tolist()
        print(f"\n   Segmento {seg}:")
        print(f"      • Tamaño: {segment_counts[seg]} clientes")
        print(f"      • Perfil: Alto gasto en {', '.join(top_categories)}")
        if seg in df[df['Segment_KMeans']==seg]['Channel_Name'].mode().values:
            print(f"      • Canal predominante: {df[df['Segment_KMeans']==seg]['Channel_Name'].mode()[0]}")

## 6. Clustering Jerárquico (Hierarchical Clustering)

El clustering jerárquico permite visualizar la estructura anidada de los clusters mediante un dendrograma.

In [ ]:
print("="*60)
print("📊 CLUSTERING JERÁRQUICO")
print("="*60)

# Calcular matriz de enlace
print("\n🔍 Calculando matriz de enlace (puede tomar unos segundos)...")
linked = linkage(X_scaled, method='ward')

# Dendrograma
plt.figure(figsize=(15, 8))
dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=True)
plt.title('Dendrograma - Clustering Jerárquico')
plt.xlabel('Índice del cliente')
plt.ylabel('Distancia')
plt.axhline(y=10, color='r', linestyle='--', label='Corte para k=4')
plt.legend()
plt.tight_layout()
plt.show()

# Cortar el dendrograma para obtener clusters
# Usar el mismo número de clusters que K-Means para comparación
hierarchical = AgglomerativeClustering(n_clusters=best_k, linkage='ward')
hierarchical_labels = hierarchical.fit_predict(X_scaled)
df['Segment_Hierarchical'] = hierarchical_labels

hier_silhouette = silhouette_score(X_scaled, hierarchical_labels)
print(f"\n📊 Resultados del Clustering Jerárquico (k={best_k}):")
print(f"   • Silhouette Score: {hier_silhouette:.4f}")

# Distribución de clientes
print(f"\n📊 Distribución de clientes por cluster:")
hier_counts = df['Segment_Hierarchical'].value_counts().sort_index()
for seg, count in hier_counts.items():
    print(f"   • Cluster {seg}: {count} clientes ({count/len(df)*100:.1f}%)")

## 7. DBSCAN Clustering

DBSCAN agrupa clientes basados en densidad, detectando automáticamente outliers.

In [ ]:
print("="*60)
print("📊 DBSCAN CLUSTERING")
print("="*60)

# Probar diferentes parámetros para DBSCAN
eps_values = [0.3, 0.5, 0.7, 1.0, 1.2, 1.5]
min_samples_values = [3, 5, 7, 10]

print("\n🔍 Probando diferentes parámetros...")
best_score = -1
best_params = None
best_n_clusters = 0
best_noise = 0
best_dbscan_labels = None

for eps in eps_values:
    for min_samples in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
        labels = dbscan.fit_predict(X_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        if n_clusters > 1 and n_clusters < 10:
            try:
                score = silhouette_score(X_scaled[labels != -1], labels[labels != -1])
                if score > best_score:
                    best_score = score
                    best_params = (eps, min_samples)
                    best_n_clusters = n_clusters
                    best_noise = n_noise
                    best_dbscan_labels = labels
                print(f"   eps={eps}, min_samples={min_samples}: clusters={n_clusters}, "
                      f"ruido={n_noise} ({n_noise/len(X_scaled)*100:.1f}%), silhouette={score:.4f}")
            except:
                pass

if best_params:
    print(f"\n✅ Mejores parámetros: eps={best_params[0]}, min_samples={best_params[1]}")
    df['Segment_DBSCAN'] = best_dbscan_labels
    
    print(f"\n📊 Resultados de DBSCAN:")
    print(f"   • Número de clusters: {best_n_clusters}")
    print(f"   • Puntos como ruido: {best_noise} ({best_noise/len(X_scaled)*100:.1f}%)")
    print(f"   • Silhouette Score: {best_score:.4f}")
    
    print(f"\n📊 Distribución de clientes por cluster:")
    for i in range(best_n_clusters):
        count = list(best_dbscan_labels).count(i)
        print(f"   • Cluster {i}: {count} clientes ({count/len(df)*100:.1f}%)")
    print(f"   • Ruido (-1): {best_noise} clientes ({best_noise/len(df)*100:.1f}%)")
else:
    print("   No se encontraron parámetros adecuados para DBSCAN")

## 8. Visualización de Clusters con PCA y t-SNE

Visualizamos los clusters encontrados por los diferentes algoritmos.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# K-Means con PCA
scatter1 = axes[0, 0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='tab10', alpha=0.7, s=50)
axes[0, 0].set_title(f'K-Means (PCA) - k={best_k}')
axes[0, 0].set_xlabel('PC1')
axes[0, 0].set_ylabel('PC2')
plt.colorbar(scatter1, ax=axes[0, 0], label='Segmento')

# K-Means con t-SNE
scatter2 = axes[0, 1].scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, cmap='tab10', alpha=0.7, s=50)
axes[0, 1].set_title(f'K-Means (t-SNE) - k={best_k}')
axes[0, 1].set_xlabel('t-SNE 1')
axes[0, 1].set_ylabel('t-SNE 2')
plt.colorbar(scatter2, ax=axes[0, 1], label='Segmento')

# Clustering Jerárquico con PCA
scatter3 = axes[1, 0].scatter(X_pca[:, 0], X_pca[:, 1], c=hierarchical_labels, cmap='tab10', alpha=0.7, s=50)
axes[1, 0].set_title(f'Clustering Jerárquico (PCA) - k={best_k}')
axes[1, 0].set_xlabel('PC1')
axes[1, 0].set_ylabel('PC2')
plt.colorbar(scatter3, ax=axes[1, 0], label='Cluster')

# DBSCAN con PCA
if best_params:
    colors = plt.cm.tab10(np.linspace(0, 1, best_n_clusters))
    for i in range(best_n_clusters):
        mask = best_dbscan_labels == i
        axes[1, 1].scatter(X_pca[mask, 0], X_pca[mask, 1], c=[colors[i]], alpha=0.7, s=50, label=f'Cluster {i}')
    noise_mask = best_dbscan_labels == -1
    axes[1, 1].scatter(X_pca[noise_mask, 0], X_pca[noise_mask, 1], c='gray', alpha=0.5, s=50, label='Ruido')
    axes[1, 1].set_title(f'DBSCAN (PCA) - eps={best_params[0]}, min_samples={best_params[1]}')
    axes[1, 1].legend()
else:
    axes[1, 1].text(0.5, 0.5, 'DBSCAN no generó clusters válidos', 
                     ha='center', va='center', transform=axes[1, 1].transAxes)
    axes[1, 1].set_title('DBSCAN')

axes[1, 1].set_xlabel('PC1')
axes[1, 1].set_ylabel('PC2')

plt.tight_layout()
plt.show()

## 9. Comparación de Métodos

Resumen de los resultados de los diferentes algoritmos de clustering.

In [ ]:
# Crear tabla comparativa
comparison_data = []

# K-Means
comparison_data.append({
    'Algoritmo': 'K-Means',
    'Parámetros': f'k={best_k}',
    'N° Clusters': best_k,
    'Silhouette': silhouette_score(X_scaled, kmeans_labels),
    'Ruido (%)': 0,
    'Interpretación': 'Segmentos basados en centroides'
})

# Clustering Jerárquico
comparison_data.append({
    'Algoritmo': 'Clustering Jerárquico',
    'Parámetros': f'k={best_k}, linkage=ward',
    'N° Clusters': best_k,
    'Silhouette': hier_silhouette,
    'Ruido (%)': 0,
    'Interpretación': 'Estructura jerárquica de clusters'
})

# DBSCAN
if best_params:
    comparison_data.append({
        'Algoritmo': 'DBSCAN',
        'Parámetros': f'eps={best_params[0]}, min_samples={best_params[1]}',
        'N° Clusters': best_n_clusters,
        'Silhouette': best_score,
        'Ruido (%)': best_noise/len(X_scaled)*100,
        'Interpretación': 'Segmentos por densidad + detección de outliers'
    })

comparison_df = pd.DataFrame(comparison_data)

print("="*60)
print("📊 TABLA COMPARATIVA DE ALGORITMOS")
print("="*60)
display(comparison_df.round(4))

## 10. Simulación - Clasificación de un Nuevo Cliente

Simulamos la segmentación de un nuevo cliente basado en sus patrones de gasto.

In [ ]:
print("="*60)
print("🔮 SIMULACIÓN: NUEVO CLIENTE")
print("="*60)

# Datos de un nuevo cliente (valores típicos)
nuevo_cliente = pd.DataFrame([[
    15000,   # Fresh (productos frescos)
    8000,    # Milk (lácteos)
    12000,   # Grocery (abarrotes)
    3000,    # Frozen (congelados)
    5000,    # Detergents_Paper (detergentes y papel)
    2000     # Delicatessen
]], columns=features)

# Escalar
nuevo_cliente_scaled = scaler.transform(nuevo_cliente)

# Predecir segmento con K-Means
segmento_kmeans = kmeans.predict(nuevo_cliente_scaled)[0]

# Mostrar información del cliente
print("\n📋 Datos del nuevo cliente (gasto anual):")
for cat in features:
    print(f"   • {cat}: {nuevo_cliente[cat].values[0]:,.0f}")

print("\n🔮 Resultado de la segmentación:")
print("-" * 50)
print(f"   • K-Means:  Segmento {segmento_kmeans}")

# Mostrar perfil del segmento asignado
if segmento_kmeans in segment_profiles.index:
    print(f"\n📊 Perfil del Segmento {segmento_kmeans}:")
    for cat in features:
        avg = segment_profiles.loc[segmento_kmeans, cat]
        print(f"   • {cat}: {avg:.0f} (promedio del segmento)")

# Recomendaciones de negocio
print("\n" + "="*60)
print("🏆 RECOMENDACIONES DE NEGOCIO:")
print("="*60)

if segmento_kmeans == 0:
    print("   • Segmento: Clientes de alto gasto en lácteos y abarrotes")
    print("   • Estrategia: Ofrecer descuentos en productos complementarios")
    print("   • Promoción: Combos de lácteos + abarrotes")
elif segmento_kmeans == 1:
    print("   • Segmento: Clientes enfocados en productos frescos")
    print("   • Estrategia: Promociones en temporada de frutas y verduras")
    print("   • Promoción: Descuentos por volumen en productos frescos")
elif segmento_kmeans == 2:
    print("   • Segmento: Clientes de alto gasto en detergentes y papel")
    print("   • Estrategia: Programas de fidelización con envíos programados")
    print("   • Promoción: Suscripción mensual con descuento")
else:
    print("   • Segmento: Perfil mixto con gasto balanceado")
    print("   • Estrategia: Ofertas cruzadas entre categorías")
    print("   • Promoción: Puntos acumulables en todas las categorías")

print("="*60)

print("\n📖 INTERPRETACIÓN DE NEGOCIO:")
print("   • La segmentación permite personalizar estrategias de marketing")
print("   • Clientes de alto valor pueden recibir atención preferencial")
print("   • Las promociones personalizadas aumentan la conversión")
print("   • El clustering jerárquico revela relaciones anidadas entre segmentos")
print("   • t-SNE ayuda a visualizar mejor la separación natural de los datos")

## 11. Conclusiones

**Resumen de resultados:**

1. **K-Means**:
   - Encontró segmentos de clientes basados en patrones de gasto
   - Ventaja: Rápido, fácil de interpretar, segmentos balanceados
   - Desventaja: Requiere conocer k, sensible a outliers

2. **Clustering Jerárquico**:
   - Proporciona un dendrograma que muestra la estructura anidada
   - Ventaja: No requiere k a priori, visualización informativa
   - Desventaja: Costoso computacionalmente para grandes datasets

3. **DBSCAN**:
   - Detecta clientes atípicos (outliers) que no encajan en segmentos
   - Ventaja: No requiere k, detecta outliers automáticamente
   - Desventaja: Sensible a parámetros, puede dejar muchos puntos sin cluster

4. **PCA y t-SNE**:
   - PCA: Reducción lineal, rápida, buena para interpretación
   - t-SNE: Reducción no lineal, mejor para visualizar clusters naturales

**Segmentos de clientes identificados:**
- Los diferentes algoritmos muestran patrones consistentes
- La elección del algoritmo depende del objetivo del negocio

**Contexto de negocio:**
- Una empresa de distribución puede usar estos segmentos para:
  * Ofrecer promociones personalizadas por segmento
  * Identificar clientes de alto valor para atención preferencial
  * Optimizar el inventario según los patrones de cada segmento

**Próximos pasos:**
- Validar segmentos con stakeholders del negocio
- Implementar campañas de marketing segmentadas
- Monitorear la evolución de los segmentos en el tiempo
- Experimentar con diferentes parámetros en t-SNE para mejor visualización

---
**Fin de la Semana 09 - Segmentación de Clientes con Múltiples Algoritmos**